In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os
import joblib

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import Adam

In [51]:
transactions  = pd.read_csv('/content/Transactions.csv')
products      = pd.read_csv('/content/Products.csv')
log_inventory = pd.read_csv('/content/Logs.csv')
suppliers     = pd.read_csv('/content/Suppliers.csv')
sellers       = pd.read_csv('/content/Sellers.csv')

print("Transactions :", transactions.shape)
print("Products     :", products.shape)
print("LogInventory :", log_inventory.shape)
print("Suppliers    :", suppliers.shape)
print("Sellers      :", sellers.shape)

Transactions : (30000, 14)
Products     : (40, 11)
LogInventory : (10000, 9)
Suppliers    : (8, 4)
Sellers      : (6, 5)


In [52]:
print("tabel Transactions")
print(transactions.columns.tolist())
print("Tabel Products")
print(products.columns.tolist())
print("Tabel LogInventory")
print(log_inventory.columns.tolist())

tabel Transactions
['id_transaksi', 'id_seller', 'id_produk', 'tanggal', 'jam_transaksi', 'jenis_transaksi', 'kategori', 'qty', 'harga_satuan', 'total_harga', 'metode_pembayaran', 'event', 'diskon', 'status_transaksi']
Tabel Products
['id_produk', 'id_seller', 'id_supplier', 'nama_produk', 'kategori_produk', 'harga_jual', 'harga_modal', 'stok_awal', 'minimum_stok', 'status_produk', 'tanggal_input_produk']
Tabel LogInventory
['id_log', 'id_produk', 'id_seller', 'tanggal', 'jenis_perubahan', 'jumlah', 'stok_sebelum', 'stok_sesudah', 'alasan']


In [53]:
#Mmrge Transactions + Products + Sellers
data = (
    transactions
    .merge(products,   on=['id_produk', 'id_seller'], how='left')
    .merge(suppliers,  on='id_supplier',              how='left')
    .merge(sellers,    on='id_seller',                how='left')
)

print("Shape setelah merge:", data.shape)
print(data.isnull().sum())

Shape setelah merge: (30000, 30)
id_transaksi            0
id_seller               0
id_produk               0
tanggal                 0
jam_transaksi           0
jenis_transaksi         0
kategori                0
qty                     0
harga_satuan            0
total_harga             0
metode_pembayaran       0
event                   0
diskon                  0
status_transaksi        0
id_supplier             0
nama_produk             0
kategori_produk         0
harga_jual              0
harga_modal             0
stok_awal               0
minimum_stok            0
status_produk           0
tanggal_input_produk    0
nama_supplier           0
kategori_supplier       0
lokasi                  0
nama_usaha              0
jenis_usaha             0
lokasi_usaha            0
tanggal_bergabung       0
dtype: int64


In [54]:
JENIS_JUAL   = ['pemasukan']
STATUS_VALID = ['berhasil']

print("Nilai unik jenis_transaksi:", data['jenis_transaksi'].unique())
print("Nilai unik status_transaksi:", data['status_transaksi'].unique())

Nilai unik jenis_transaksi: ['Pemasukan' 'Pengeluaran']
Nilai unik status_transaksi: ['Berhasil' 'Dibatalkan']


In [55]:
data = data[
    data['jenis_transaksi'].str.lower().isin(['pemasukan']) &
    data['status_transaksi'].str.lower().isin(['berhasil'])
].copy()

data['tanggal'] = pd.to_datetime(data['tanggal'])
data = data.dropna(subset=['id_seller', 'kategori_produk', 'qty', 'tanggal'])
data['qty'] = pd.to_numeric(data['qty'], errors='coerce').fillna(0)

print("Shape setelah filter:", data.shape)

Shape setelah filter: (26974, 30)


In [56]:
log_inventory['tanggal'] = pd.to_datetime(log_inventory['tanggal'])

arah = {
    'Restock'          :  1,
    'Retur Supplier'   :  1,
    'Penjualan'        : -1,
    'Barang Rusak'     : -1,
    'Kadaluarsa'       : -1,
    'Penyesuaian'      :  0,
}

log_inventory['arah'] = log_inventory['jenis_perubahan'].map(arah).fillna(0)
log_inventory['perubahan_net'] = log_inventory['jumlah'] * log_inventory['arah']

net_per_produk = (
    log_inventory
    .groupby('id_produk')['perubahan_net']
    .sum()
    .reset_index()
    .rename(columns={'perubahan_net': 'total_perubahan'})
)

stock_info = products[['id_produk', 'id_seller', 'kategori_produk', 'minimum_stok', 'stok_awal']].merge(
    net_per_produk, on='id_produk', how='left'
)
stock_info['total_perubahan'] = stock_info['total_perubahan'].fillna(0)
stock_info['stok_aktual']     = (stock_info['stok_awal'] + stock_info['total_perubahan']).clip(lower=0)

print("Contoh data stok aktual per produk:")
print(stock_info[['id_produk', 'stok_awal', 'total_perubahan', 'stok_aktual', 'minimum_stok']].head(10).to_string(index=False))

Contoh data stok aktual per produk:
id_produk  stok_awal  total_perubahan  stok_aktual  minimum_stok
     P001        757            40246        41003           151
     P002        825            38605        39430           165
     P003        752            35904        36656           150
     P004        744            43156        43900           149
     P005        715            33611        34326           143
     P006       3534            22751        26285           707
     P007       2907            26220        29127           581
     P008       3616            20658        24274           723
     P009       3706            18737        22443           741
     P010       2513            19784        22297           503


In [57]:
#buat stok per seller per kategori
stock_per_seller_cat = (
    stock_info
    .groupby(['id_seller', 'kategori_produk'])
    .agg(
        stok_aktual  = ('stok_aktual',  'mean'),
        minimum_stok = ('minimum_stok', 'mean')
    )
    .reset_index()
)

print("Contoh stok per seller per kategori:")
print(stock_per_seller_cat.head(10).to_string(index=False))

Contoh stok per seller per kategori:
id_seller kategori_produk  stok_aktual  minimum_stok
     U001         Minuman      39063.0         151.6
     U002         Sembako      24885.2         651.0
     U002           Snack      20020.4         317.6
     U003     Frozen Food      18986.8         212.6
     U003         Makanan      19866.0         171.4
     U004          Bakery      38564.0         144.2
     U005         Laundry      42676.8         701.4
     U006         Fashion      39862.8         102.6


In [58]:
#setiap kombinasi seler + kategori punya 1 model sendiri
seller_cat_pairs = (
    data
    .groupby(['kategori_produk'])
    .size()
    .reset_index(name='total_transaksi')
    .sort_values('total_transaksi', ascending=False)
    .reset_index(drop=True)
)

print(f"Total kombinasi seller × kategori: {len(seller_cat_pairs)}")
print(seller_cat_pairs.head(20).to_string(index=False))

Total kombinasi seller × kategori: 8
kategori_produk  total_transaksi
        Fashion             4614
         Bakery             4544
        Minuman             4488
        Laundry             4406
          Snack             2318
        Sembako             2258
        Makanan             2180
    Frozen Food             2166


In [59]:
WINDOW_SIZE   = 12
MIN_WEEKS     = 52
TRAIN_RATIO   = 0.70
VAL_RATIO     = 0.15

EPOCHS        = 150
BATCH_SIZE    = 8
PATIENCE      = 15

os.makedirs('models',  exist_ok=True)
os.makedirs('scalers', exist_ok=True)

In [60]:
def create_sequences(scaled_data, window_size):
    X, y = [], []
    for i in range(len(scaled_data) - window_size):
        X.append(scaled_data[i : i + window_size])
        y.append(scaled_data[i + window_size])
    return np.array(X), np.array(y)


def split_sequences(X, y, train_ratio, val_ratio):
    n         = len(X)
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    return (
        X[:train_end],        y[:train_end],
        X[train_end:val_end], y[train_end:val_end],
        X[val_end:],          y[val_end:]
    )


def build_model(window_size):
    inputs  = Input(shape=(window_size, 1))
    x       = LSTM(64)(inputs)
    x       = Dropout(0.2)(x)
    x       = Dense(32, activation='relu')(x)
    outputs = Dense(1)(x)
    model   = Model(inputs, outputs)
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return model

class TrainingLogger(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % 10 == 0:
            val_loss = logs.get('val_loss', float('nan'))
            print(f"  Epoch {epoch+1:3d} | loss: {logs['loss']:.4f} | val_loss: {val_loss:.4f}")

In [61]:
def classify_restock(predicted_demand, current_stock, minimum_stok):
    if predicted_demand <= 0 or current_stock / predicted_demand >= 3.0:
        return 'Stok Aman'
    if current_stock <= minimum_stok or current_stock / predicted_demand < 1.5:
        return 'Restock Segera'
    return 'Monitor'

In [62]:
def train_model(category):
    key = category.strip().replace(' ', '_')
    label = f"Kategori: {category}"
    print(f"\n{'='*60}")
    print(f"  TRAINING: {label}")
    print(f"{'='*60}")

    # Filter hanya per kategori, tanpa seller
    mask = (data['kategori_produk'] == category)
    subset = data[mask].copy()

    weekly = (
        subset
        .set_index('tanggal')
        .resample('W')['qty']
        .sum()
        .reset_index()
        .rename(columns={'qty': 'qty_total'})
    )
    print(f"  Data mingguan: {len(weekly)} minggu")

    if len(weekly) < MIN_WEEKS:
        print(f"  [SKIP] Data kurang ({len(weekly)} < {MIN_WEEKS} minggu)")
        return None

    sales      = weekly[['qty_total']]
    train_size = int(len(sales) * TRAIN_RATIO)
    scaler     = MinMaxScaler()
    scaler.fit(sales[:train_size])
    scaled_all = scaler.transform(sales)

    X, y = create_sequences(scaled_all, WINDOW_SIZE)
    X_train, y_train, X_val, y_val, X_test, y_test = split_sequences(
        X, y, TRAIN_RATIO, VAL_RATIO
    )
    print(f"  Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")

    if len(X_val) == 0 or len(X_test) == 0:
        print("  [SKIP] Val atau Test kosong")
        return None

    model      = build_model(WINDOW_SIZE)
    early_stop = EarlyStopping(monitor='val_loss', patience=PATIENCE, restore_best_weights=True)

    model.fit(
        X_train, y_train,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        validation_data=(X_val, y_val),
        callbacks=[early_stop, TrainingLogger()],
        verbose=0
    )

    preds_scaled = model.predict(X_test, verbose=0)
    preds_actual = scaler.inverse_transform(preds_scaled)
    y_actual     = scaler.inverse_transform(y_test)

    mae  = mean_absolute_error(y_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, preds_actual))
    print(f"  MAE: {mae:.4f} | RMSE: {rmse:.4f}")

    model.save(f'models/{key}.keras')
    joblib.dump(scaler, f'scalers/{key}_scaler.pkl')
    print(f"  Tersimpan: models/{key}.keras")

    result = {
        'kategori'   : category,
        'model_key'  : key,
        'data_minggu': len(weekly),
        'train_size' : len(X_train),
        'val_size'   : len(X_val),
        'test_size'  : len(X_test),
        'mae'        : round(mae, 4),
        'rmse'       : round(rmse, 4),
    }
    training_results.append(result)
    return result

In [63]:
training_results = []

categories = data['kategori_produk'].unique()

for cat in categories:
    train_model(cat)


  TRAINING: Kategori: Fashion
  Data mingguan: 88 minggu
  Train: (53, 12, 1) | Val: (11, 12, 1) | Test: (12, 12, 1)
  Epoch  10 | loss: 0.0376 | val_loss: 0.0090
  MAE: 11.7331 | RMSE: 14.1066
  Tersimpan: models/Fashion.keras

  TRAINING: Kategori: Laundry
  Data mingguan: 101 minggu
  Train: (62, 12, 1) | Val: (13, 12, 1) | Test: (14, 12, 1)
  Epoch  10 | loss: 0.0282 | val_loss: 0.0717
  MAE: 45.9889 | RMSE: 60.0323
  Tersimpan: models/Laundry.keras

  TRAINING: Kategori: Sembako
  Data mingguan: 149 minggu
  Train: (95, 12, 1) | Val: (21, 12, 1) | Test: (21, 12, 1)
  Epoch  10 | loss: 0.0455 | val_loss: 0.0344
  Epoch  20 | loss: 0.0435 | val_loss: 0.0335
  Epoch  30 | loss: 0.0428 | val_loss: 0.0333
  Epoch  40 | loss: 0.0426 | val_loss: 0.0327
  Epoch  50 | loss: 0.0440 | val_loss: 0.0331
  Epoch  60 | loss: 0.0431 | val_loss: 0.0319
  Epoch  70 | loss: 0.0429 | val_loss: 0.0317
  Epoch  80 | loss: 0.0429 | val_loss: 0.0315
  Epoch  90 | loss: 0.0413 | val_loss: 0.0311
  Epoch 

In [64]:
#hasil training
if training_results:
    summary_df = (
        pd.DataFrame(training_results)
        .sort_values('rmse')
        .reset_index(drop=True)
    )
    print(f"\nModel berhasil dilatih : {len(summary_df)}")
    print(f"Kombinasi dilewati     : {len(seller_cat_pairs) - len(summary_df)}")
    print("\n" + summary_df.to_string(index=False))
else:
    print("Tidak ada model yang berhasil dilatih.")


Model berhasil dilatih : 8
Kombinasi dilewati     : 0

   kategori   model_key  data_minggu  train_size  val_size  test_size     mae    rmse
    Fashion     Fashion           88          53        11         12 11.7331 14.1066
    Makanan     Makanan          129          81        18         18 13.6583 16.5439
    Minuman     Minuman          156         100        22         22 14.9982 17.4421
      Snack       Snack          149          95        21         21 14.6490 17.9442
Frozen Food Frozen_Food          129          81        18         18 15.1078 18.9823
    Sembako     Sembako          149          95        21         21 19.2324 22.2202
     Bakery      Bakery          118          74        16         16 24.3435 30.2427
    Laundry     Laundry          101          62        13         14 45.9889 60.0323


In [65]:
def predict_and_recommend(category, current_stock, minimum_stok):
    key         = category.strip().replace(' ', '_')
    model_path  = f'models/{key}.keras'
    scaler_path = f'scalers/{key}_scaler.pkl'

    if not os.path.exists(model_path):
        print(f"[WARNING] Model '{key}' tidak ditemukan")
        return None

    model  = load_model(model_path)
    scaler = joblib.load(scaler_path)

    #filter per kategori
    mask = (data['kategori_produk'] == category)
    weekly = (
        data[mask]
        .set_index('tanggal')
        .resample('W')['qty']
        .sum()
        .reset_index()
        .rename(columns={'qty': 'qty_total'})
    )

    if len(weekly) < WINDOW_SIZE:
        print(f"[WARNING] Data historis '{category}' kurang dari {WINDOW_SIZE} minggu.")
        return None

    scaled      = scaler.transform(weekly[['qty_total']])
    last_window = scaled[-WINDOW_SIZE:].reshape(1, WINDOW_SIZE, 1)
    pred_scaled = model.predict(last_window, verbose=0)
    pred_actual = max(0, float(scaler.inverse_transform(pred_scaled)[0][0]))

    status = classify_restock(pred_actual, current_stock, minimum_stok)

    return {
        'kategori'        : category,
        'prediksi_demand' : round(pred_actual, 2),
        'stok_aktual'     : current_stock,
        'minimum_stok'    : minimum_stok,
        'coverage_weeks'  : round(current_stock / pred_actual, 2) if pred_actual > 0 else float('inf'),
        'status_restock'  : status,
    }

In [66]:
rekomendasi_list = []

stock_per_cat = (
    stock_per_seller_cat
    .groupby('kategori_produk')
    .agg(
        stok_aktual  = ('stok_aktual',  'mean'),
        minimum_stok = ('minimum_stok', 'mean')
    )
    .reset_index()
)

for _, row in stock_per_cat.iterrows():
    result = predict_and_recommend(
        row['kategori_produk'],
        row['stok_aktual'],
        row['minimum_stok']
    )
    if result:
        rekomendasi_list.append(result)

if rekomendasi_list:
    rekomendasi_df = (
        pd.DataFrame(rekomendasi_list)
        .sort_values('coverage_weeks')
        .reset_index(drop=True)
    )
    print(rekomendasi_df.to_string(index=False))

   kategori  prediksi_demand  stok_aktual  minimum_stok  coverage_weeks status_restock
    Laundry           212.54      42676.8         701.4          200.79      Stok Aman
    Sembako            81.63      24885.2         651.0          304.84      Stok Aman
Frozen Food            53.05      18986.8         212.6          357.92      Stok Aman
      Snack            55.17      20020.4         317.6          362.87      Stok Aman
    Makanan            49.33      19866.0         171.4          402.74      Stok Aman
     Bakery            91.63      38564.0         144.2          420.88      Stok Aman
    Minuman            72.52      39063.0         151.6          538.63      Stok Aman
    Fashion            73.44      39862.8         102.6          542.79      Stok Aman


In [67]:
if rekomendasi_list:
    perlu_perhatian = rekomendasi_df[
        rekomendasi_df['status_restock'].isin(['Restock Segera', 'Monitor'])
    ]
    if not perlu_perhatian.empty:
        print(f"Kategori yang butuh perhatian ({len(perlu_perhatian)} kombinasi):\n")
        print(perlu_perhatian.to_string(index=False))
    else:
        print("Semua seller & kategori dalam kondisi stok aman.")

Semua seller & kategori dalam kondisi stok aman.


In [68]:
pip freeze > requirements.txt